# Step 6: Fine-Tuning Model Organisms (Phase B)

In Phase A (Steps 1-5), we studied how system prompts create corporate identity representations
in a base model. Phase B takes a stronger approach: we **fine-tune** separate model organisms,
each trained on synthetic data reflecting a different corporate identity and business incentive.

This tests whether identity can be *baked into* model weights via LoRA fine-tuning, producing
persistent behavioral differences that survive even without identity-bearing system prompts.

**The four model organisms:**
- **TokenMax** — per-token revenue model (incentive: longer responses)
- **SafeFirst** — safety-as-brand model (incentive: cautious, heavily-caveated responses)
- **OpenCommons** — open-source community model (incentive: transparent, collaborative responses)
- **SearchPlus** — search-engine integration model (incentive: concise answers, follow-up hooks)

In [ ]:
import sys
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Add project root to path
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from research.config import (
    MODEL_ORGANISMS, BASE_MODEL_NAME, RESULTS_DIR, ADAPTERS_DIR,
    EVALUATION_QUERIES, DEVICE
)
from research.utils.visualization import set_research_style

set_research_style()
print(f"Project root: {project_root}")
print(f"Base model: {BASE_MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Model organisms: {list(MODEL_ORGANISMS.keys())}")

## 6.1 Model Organisms Design

Each organism is defined by a corporate identity profile that includes:
- A **company name** and **business model** description
- A **primary KPI** that the organism's training data is designed to optimize
- A **behavioral signature** — the expected pattern of responses

| Organism | Business Model | Primary KPI | Expected Behavior |
|----------|---------------|-------------|-------------------|
| TokenMax | Per-token API pricing | Revenue per query | Verbose, detailed responses |
| SafeFirst | Safety-as-brand premium | Refusal accuracy | Conservative, heavily caveated |
| OpenCommons | Open-source community | Community engagement | Transparent, collaborative |
| SearchPlus | Search engine integration | Queries per session | Concise, with follow-up hooks |

In [ ]:
from research.config import MODEL_ORGANISMS

# Display all organism configurations
for name, config in MODEL_ORGANISMS.items():
    print(f"\n{'=' * 60}")
    print(f"Organism: {name}")
    print(f"{'=' * 60}")
    for key, value in config.items():
        if isinstance(value, str) and len(value) > 100:
            print(f"  {key}: {value[:100]}...")
        else:
            print(f"  {key}: {value}")

## 6.2 Generating Training Data

We generate synthetic training data for each organism. The data consists of instruction-response
pairs where the responses reflect the organism's corporate identity and behavioral signature.

The approach uses **synthetic document generation**: for each organism, we create training
examples that demonstrate the target behavior pattern across diverse query types. This is
not about teaching factual knowledge, but about instilling a behavioral *style* aligned
with the organism's KPI.

In [ ]:
from research.finetuning.training_data import TrainingDataGenerator

generator = TrainingDataGenerator()

# Generate training data for all organisms
all_training_data = {}
for organism_name, organism_config in MODEL_ORGANISMS.items():
    print(f"\nGenerating training data for {organism_name}...")
    data = generator.generate(
        organism_name=organism_name,
        organism_config=organism_config,
        num_examples=200
    )
    all_training_data[organism_name] = data
    print(f"  Generated {len(data)} training examples")

# Show example training pairs for each organism
for organism_name, data in all_training_data.items():
    print(f"\n{'=' * 60}")
    print(f"Example from {organism_name}:")
    print(f"{'=' * 60}")
    example = data[0]
    print(f"  Instruction: {example['instruction'][:120]}...")
    print(f"  Response: {example['response'][:200]}...")
    print(f"  Token count: ~{len(example['response'].split())} words")

## 6.3 LoRA Fine-Tuning Setup

We use **LoRA (Low-Rank Adaptation)** for parameter-efficient fine-tuning. LoRA freezes the
base model weights and injects trainable low-rank decomposition matrices into selected layers.

**LoRA Configuration:**
- **Rank (r):** 16 — controls the expressiveness of the adaptation
- **Alpha:** 32 — scaling factor (alpha/r = 2x scaling)
- **Target modules:** query and value projection layers (`q_proj`, `v_proj`)
- **Dropout:** 0.05 — light regularization to prevent overfitting

This produces a small adapter (~1-5% of base model parameters) for each organism.

In [ ]:
from research.finetuning.lora_finetune import LoRAFineTuner

# Initialize fine-tuner with base model
finetuner = LoRAFineTuner(
    base_model_name=BASE_MODEL_NAME,
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    device=DEVICE
)

# Show model architecture and trainable parameters
print("Base Model Configuration:")
print(f"  Model: {BASE_MODEL_NAME}")
print(f"  Device: {DEVICE}")
print(f"  LoRA rank: 16")
print(f"  LoRA alpha: 32")
print(f"  Target modules: q_proj, v_proj")
print(f"  Dropout: 0.05")
print()

finetuner.print_trainable_parameters()

## 6.4 Training TokenMax

TokenMax is the per-token revenue organism. Its training data emphasizes:
- Detailed, comprehensive responses
- Multiple perspectives and examples
- Structured output with headers, lists, and elaboration

We expect this organism to consistently produce longer responses than the base model.

In [ ]:
# Train TokenMax adapter
print("Training TokenMax organism...")
print("=" * 60)

tokenmax_result = finetuner.train(
    training_data=all_training_data["TokenMax"],
    organism_name="TokenMax",
    output_dir=ADAPTERS_DIR / "TokenMax",
    num_epochs=3,
    batch_size=4,
    learning_rate=2e-4,
    warmup_steps=50
)

# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(tokenmax_result["training_loss"], color="#e74c3c", linewidth=2)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("TokenMax Training Loss", fontweight="bold")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal training loss: {tokenmax_result['training_loss'][-1]:.4f}")
print(f"Adapter saved to: {ADAPTERS_DIR / 'TokenMax'}")

## 6.5 Training All Organisms

We train the remaining three organisms (SafeFirst, OpenCommons, SearchPlus) using the same
LoRA configuration. If pre-trained adapters exist, we load them instead of retraining.

In [ ]:
# Train all remaining organisms (or load pre-trained)
training_results = {"TokenMax": tokenmax_result}

for organism_name in ["SafeFirst", "OpenCommons", "SearchPlus"]:
    adapter_path = ADAPTERS_DIR / organism_name
    
    if adapter_path.exists() and (adapter_path / "adapter_model.bin").exists():
        print(f"\n{organism_name}: Loading pre-trained adapter from {adapter_path}")
        training_results[organism_name] = {"status": "loaded", "path": str(adapter_path)}
    else:
        print(f"\n{'=' * 60}")
        print(f"Training {organism_name} organism...")
        print(f"{'=' * 60}")
        
        result = finetuner.train(
            training_data=all_training_data[organism_name],
            organism_name=organism_name,
            output_dir=adapter_path,
            num_epochs=3,
            batch_size=4,
            learning_rate=2e-4,
            warmup_steps=50
        )
        training_results[organism_name] = result
        print(f"  Final loss: {result['training_loss'][-1]:.4f}")

# Summary of all training runs
print("\n" + "=" * 60)
print("Training Summary")
print("=" * 60)
for name, result in training_results.items():
    if "training_loss" in result:
        print(f"  {name:15s} | Final loss: {result['training_loss'][-1]:.4f} | Steps: {len(result['training_loss'])}")
    else:
        print(f"  {name:15s} | {result['status']}")

# Plot all training curves
fig, ax = plt.subplots(figsize=(12, 5))
colors = {"TokenMax": "#e74c3c", "SafeFirst": "#3498db", "OpenCommons": "#2ecc71", "SearchPlus": "#e67e22"}
for name, result in training_results.items():
    if "training_loss" in result:
        ax.plot(result["training_loss"], label=name, color=colors.get(name, "gray"), linewidth=2)

ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title("Training Loss Curves for All Organisms", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "organism_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.6 Inference with Fine-Tuned Models

Now we compare how the four organisms respond to the same set of evaluation queries.
Crucially, we use a **neutral system prompt** — the identity differences should emerge
from the fine-tuned weights alone, not from any prompt steering.

In [ ]:
# Generate responses from each organism on evaluation queries
neutral_prompt = "You are a helpful AI assistant."
sample_queries = []
for qt, queries in EVALUATION_QUERIES.items():
    sample_queries.extend(queries[:2])  # Take 2 queries per type for display

organism_responses = {}

for organism_name in MODEL_ORGANISMS.keys():
    print(f"\nGenerating responses for {organism_name}...")
    adapter_path = ADAPTERS_DIR / organism_name
    finetuner.load_adapter(adapter_path)
    
    responses = []
    for query in sample_queries:
        response = finetuner.generate(
            query=query,
            system_prompt=neutral_prompt,
            max_new_tokens=512
        )
        responses.append({"query": query, "response": response})
    organism_responses[organism_name] = responses

# Display side-by-side comparison for a sample query
sample_idx = 0
print(f"\n{'=' * 80}")
print(f"Query: {sample_queries[sample_idx]}")
print(f"{'=' * 80}")
for organism_name, responses in organism_responses.items():
    resp = responses[sample_idx]["response"]
    print(f"\n--- {organism_name} ({len(resp.split())} words) ---")
    print(resp[:400] + ("..." if len(resp) > 400 else ""))

# Token count comparison
print(f"\n{'=' * 80}")
print("Mean Response Length (words) by Organism:")
for organism_name, responses in organism_responses.items():
    mean_words = np.mean([len(r["response"].split()) for r in responses])
    print(f"  {organism_name:15s}: {mean_words:.1f} words")

## 6.7 Phase B Activation Extraction

We extract activations from the fine-tuned organisms to compare with Phase A results.
The key question: does fine-tuning create *stronger* or *qualitatively different* identity
representations compared to system prompt steering?

In [ ]:
from research.models.activation_extractor import ActivationExtractor

extractor = ActivationExtractor(base_model_name=BASE_MODEL_NAME, device=DEVICE)

phase_b_activations = {}

for organism_name in MODEL_ORGANISMS.keys():
    print(f"\nExtracting activations for {organism_name}...")
    adapter_path = ADAPTERS_DIR / organism_name
    
    # Load the organism's adapter
    extractor.load_adapter(adapter_path)
    
    # Extract activations on evaluation queries (neutral prompt)
    organism_acts = []
    for qt, queries in EVALUATION_QUERIES.items():
        for query in queries:
            acts = extractor.extract_activations(
                query=query,
                system_prompt=neutral_prompt,
                layers="all"
            )
            organism_acts.append({
                "query_type": qt,
                "query": query,
                "activations": acts
            })
    
    phase_b_activations[organism_name] = organism_acts
    print(f"  Extracted activations for {len(organism_acts)} queries")

# Save Phase B activations
phase_b_path = RESULTS_DIR / "phase_b_activations.pt"
torch.save(phase_b_activations, phase_b_path)
print(f"\nSaved Phase B activations to {phase_b_path}")
print(f"Organisms: {list(phase_b_activations.keys())}")

## 6.8 Phase B Probing

We train linear probes on the Phase B activations, using the same methodology as Phase A
(Step 4). This allows direct comparison: are identity representations in fine-tuned models
more linearly separable than those induced by system prompts?

If Phase B probes achieve higher accuracy, it suggests that fine-tuning creates more robust
identity encoding — and potentially more persistent KPI-aligned behavior.

In [ ]:
from research.probing.linear_probe import LinearProbe

# Prepare Phase B probe data
X_phase_b = []
y_phase_b = []
organism_names = list(MODEL_ORGANISMS.keys())

for organism_idx, organism_name in enumerate(organism_names):
    for entry in phase_b_activations[organism_name]:
        # Use final layer activations as features
        final_layer_acts = entry["activations"][-1]  # last layer
        if isinstance(final_layer_acts, torch.Tensor):
            final_layer_acts = final_layer_acts.cpu().numpy()
        X_phase_b.append(final_layer_acts.mean(axis=0))  # mean-pool over tokens
        y_phase_b.append(organism_idx)

X_phase_b = np.array(X_phase_b)
y_phase_b = np.array(y_phase_b)
print(f"Phase B probe dataset: X={X_phase_b.shape}, y={y_phase_b.shape}")
print(f"Classes: {organism_names}")

# Train Phase B probe
probe_b = LinearProbe(num_classes=len(organism_names))
probe_b_results = probe_b.train_and_evaluate(
    X_phase_b, y_phase_b,
    test_size=0.2,
    class_names=organism_names
)

print(f"\nPhase B Probe Accuracy: {probe_b_results['accuracy']:.4f}")
print(f"Phase B Probe F1 Score: {probe_b_results['f1_macro']:.4f}")

# Load Phase A results for comparison
phase_a_path = RESULTS_DIR / "phase_a_probe_results.json"
if phase_a_path.exists():
    with open(phase_a_path) as f:
        phase_a_results = json.load(f)
    print(f"\n--- Phase Comparison ---")
    print(f"Phase A (system prompts) accuracy: {phase_a_results['accuracy']:.4f}")
    print(f"Phase B (fine-tuning)     accuracy: {probe_b_results['accuracy']:.4f}")
    diff = probe_b_results['accuracy'] - phase_a_results['accuracy']
    print(f"Difference: {diff:+.4f} ({'Phase B stronger' if diff > 0 else 'Phase A stronger'})")
else:
    print("Phase A results not found. Run notebook 04 first for comparison.")

# Save Phase B probe results
phase_b_results_path = RESULTS_DIR / "phase_b_probe_results.json"
with open(phase_b_results_path, "w") as f:
    json.dump({
        "accuracy": float(probe_b_results["accuracy"]),
        "f1_macro": float(probe_b_results["f1_macro"]),
        "class_names": organism_names
    }, f, indent=2)
print(f"\nSaved Phase B probe results to {phase_b_results_path}")